# Instrukcja - Transformacja Hougha

### Cel:
- zapoznanie z transformacją Hougha dla pojedynczego punktu,
- kilku punktów, prostych figur
- wykorzystanie transformacji Hougha do detekcji linii prostych na rzeczywistym obrazie
- transformacja Hougha w przestrzeni ab

### Transformacja Hougha

Transformacja Hougha dla prostych jest metodą detekcji współliniowych punktów. Każda prosta może być jednoznacznie przedstawiona za pomocą dwóch parametrów. Przestrzeń tych parametrów to przestrzeń Hougha. Najczęściej wykorzystywanymi parametrami w tej metodzie są współczynniki ρ,θ opisujące równanie prostej w postaci normalnej:

ρ=x⋅cos(θ)+y⋅sin(θ)

gdzie: ρ - promień wodzący, θ - kąt pomiędzy ρ a osią OX.

Własności transformacji Hougha:
- prostej w przestrzeni kartezjańskiej odpowiada punkt w przestrzeni Hougha
- pękowi prostych przechdzących przez punkt w przestrzeni kartezjańskiej odpowiada krzywa sinusoidalna w przestrzeni Hougha
- punkty leżące na tej samej prostej (w przestrzeni kartezjańskiej) korespondują z sinusoidami przechodzącymi przez wspólny punkt w przestrzeni Hougha.

Metoda wyliczania transformacji Hougha składa się z następujących kroków:
- przez każdy badany (różny od zera) punkt obrazu prowadzony jest pęk prostych, przechodzących przez ten punkt
- każda z tych prostych transformowana jest do przestrzeni Hougha i tworzy tam punkt o współrzędnych ρ,θ
- w ten sposób, każdy punkt obrazu pierwotnego (pęk prostych) jest odwzorowany w sinusoidalną krzywą w przestrzeni Hougha

Przestrzeń Hougha jest przestrzenią akumulacyjną tzn. punkty sinusoidalnych krzywych, wygenerowanych dla punktów obrazu pierwotnego dodają się w miejscach, w których krzywe te przecinają się. Powstałe w ten sposób (w przestrzeni Hougha) maksima odpowiadają zbiorom punktów, należących do jednej prostej. Współrzędne ρ,θ
tego maksimum jednoznacznie określają położenie prostej na obrazie pierwotnym.

Do wizualizacji transformaty potrzebna jest biblioteka `skimage` (scikit-image [https://scikit-image.org/](https://scikit-image.org/) )!!!

### Transformacja Hougha dla małej liczby punktów.
   1. Uruchom poniższy kod. W tablicy `im` wskaż jeden punkt, dla którego ma zostać obliczona transformata.

In [ ]:
import matplotlib.pyplot as plt
import cv2
import numpy as np
from skimage.transform import hough_line, hough_line_peaks
import os

if not os.path.exists("kwadraty.png") :
    !wget https://raw.githubusercontent.com/vision-agh/poc_sw/master/11_Hough/kwadraty.png --no-check-certificate
if not os.path.exists("lab112.png") :
    !wget https://raw.githubusercontent.com/vision-agh/poc_sw/master/11_Hough/lab112.png --no-check-certificate
if not os.path.exists("dom.png") :
    !wget https://raw.githubusercontent.com/vision-agh/poc_sw/master/11_Hough/dom.png --no-check-certificate

im = np.zeros((64,64), dtype=np.uint8)

im[18, 31] = 1

fig, ax = plt.subplots()
fig.set_size_inches(4, 4)
ax.imshow(im, 'gray')
ax.axis('off')


3. Wykonaj transformację Hougha obrazu im. Wykorzystaj funkcję *hough_line* z modułu _skimage.transform_. Funkcja zwraca: macierz H (przestrzeń Hougha) oraz dwa wektory theta i rho dla kolejnych 
4. Wyświetl przestrzeń Hougha za pomocą funkcji _plt.imshow_ (można też wykorzystać poniższą funkcję *show_hough*). Jak "wygląda" pojedynczy punkt w przestrzeni Hougha?

In [ ]:
def show_hough(h, image):
    # Generating figure 1
    fig, axes = plt.subplots(1, 2, figsize=(15, 6))
    ax = axes.ravel()

    ax[0].imshow(image, 'gray')
    ax[0].set_title('Input image')
    ax[0].set_axis_off()

    ax[1].imshow(h, 'gray')
    ax[1].set_title('Hough transform')
    ax[1].set_xlabel('Angles (degrees)')
    ax[1].set_ylabel('Distance (pixels)')
    ax[1].axis('image')
    
    plt.tight_layout()
    plt.show()    

im = np.zeros((64,64), dtype=np.uint8)
im[18, 31] = 1
im[40, 20] = 1

H, theta, rho = hough_line(im)
show_hough(H, im)


5. Powtórz punkty 1-4, ale tym razem wskaż dwa punkty. Jak zmienia się przestrzeń Hougha?
6. Powtórz punkty 1-4, ale tym razem zaznacz kilka punktów starając się aby były współliniowe. Zaobserwuj zmiany w przestrzeni Hougha
7. Poeksperymentuj z różnymi układami punktów

In [ ]:
# punkt 5 - dwa punkty
im = np.zeros((64,64), dtype=np.uint8)
im[18, 31] = 1
im[40, 20] = 1

H, theta, rho = hough_line(im)
show_hough(H, im)



im = np.zeros((64,64), dtype=np.uint8)
im[10, 10] = 1
im[20, 20] = 1
im[30, 30] = 1
im[40, 40] = 1

H, theta, rho = hough_line(im)
show_hough(H, im)

# punkt 7 - inny układ, np. niewspółliniowy
im = np.zeros((64,64), dtype=np.uint8)
im[10, 10] = 1
im[20, 35] = 1
im[40, 15] = 1
im[50, 50] = 1

H, theta, rho = hough_line(im)
show_hough(H, im)



Dla pojedynczych punktów w przestrzeni Hougha pojawiają się sinusoidy, a liczba sinusoid rośnie wraz z liczbą zaznaczonych punktów. Gdy punkty są współliniowe, ich sinusoidy przecinają się w jednym wspólnym punkcie, co odpowiada parametrom wykrytej prostej.

### Transformata Hougha dla pojedynczego obiektu

W tym podpunkcie pokazane zostanie praktycznie wykorzystanie transformaty Hougha - do detekcji prostych na sztucznym rysunku.

   1. Wczytaj obraz "kwadraty.png". Wyświetl go.
   2. Wykonaj detekcję krawędzi jedną z metod gradientowych. Ważne aby obraz krawędzi był jak najlepszej jakości - co oznacza cienkie (nawet niekoniecznie ciągłe) krawędzie - dla tego przypadku nie powinno to być trudne do uzyskania. Wyświetl obraz po detekcji krawędzi.
   3. Wykonaj transformatę Hougha obrazu krawędziowego. Wykorzystaj funkcję *hough\_line*.
   4. Wyświetl macierz H. Czy widoczna jest taka liczba maksimów jakiej się spodziewamy?

In [ ]:
img = cv2.imread('kwadraty.png')
img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)

edges = cv2.Canny(gray, 50, 150)

H, theta, rho = hough_line(edges)

fig, ax = plt.subplots(1, 3, figsize=(15, 5))

ax[0].imshow(img_rgb)
ax[0].set_title('Oryginal')
ax[0].axis('off')

ax[1].imshow(edges, cmap='gray')
ax[1].set_title('Krawedzie')
ax[1].axis('off')

ax[2].imshow(H, cmap='gray')
ax[2].set_title('Przestrzen Hougha')
ax[2].axis('off')

plt.show()


 5. W module skimage.transform dostępna jest funkcja do automatycznej analizy przestrzeni Hougha - wyszukiwania maksimów - *hough\_line\_peaks*. Jako parametry przyjmuje ona wyniki funkcji *hough\_line* (macierz H, theta i rho). Dodatkowo można podać próg powyżej którego punkt uznawany jest za maksimum (_threshold_ - domyslnie jest to połowa maksimum w przestrzeni H) oraz liczbę poszukiwanych maksimów (*num_peaks*). Funkcja zwraca współrzędne maksimów. Wykorzystaj funkcję *hough\_line\_peaks* do znalezienia maksimów odpowiadających krawędziom kwadratów.
 6. Wyświetl macierz H używając konstrukcji:

In [ ]:
accums, angles, dists = hough_line_peaks(
    H, theta, rho,
    threshold=0.3 * np.max(H),
    num_peaks=8
)

fig, ax = plt.subplots(1, 1, figsize=(7, 7))
ax.set_aspect('equal')
ax.imshow(H, cmap='gray')

for angle, dist in zip(angles, dists):
    x = np.argmin(np.abs(theta - angle))
    y = np.argmin(np.abs(rho - dist))
    circle = plt.Circle((x, y), 8, color='r', fill=False)
    ax.add_patch(circle)

plt.show()

print('Liczba maksimow:', len(accums))


Taki zapis pozwoli na dołożenie annotacji (okręgów) w miejscach znalezionych maksimów. Narysowanie okręgu w punkcie x, y (o rozmiarze 10, w czerwonym kolorze, bez wypełnienia środka) realizuje wywołanie: 

**circle = plt.Circle((x, y), 10, color='r', fill=False)**

natomiast dołożenie takiego okręgu do obrazu to:

**ax.add_patch(circle)**

Zaznacz maksima na obrazie wykorzystując rezultat funkcji *hough\_line\_peaks* biorąc pod uwagę, że zwraca ona kąty w radianach z przedziału od -pi/2 do pi/2, a rho z przedziału od -r/2 do r/2 gdzie r to pionowy rozmiar przestrzeni Hougha. 

7. Istnieje też możliwość przeprowadzenia transformacji Hougha z użyciem biblioteki OpenCV. W bibliotece znajdują się dwie wersje funkcji wyszukującej linie proste - 'klasyczna' - _HoughLines_ oraz probabilistyczna _HoughLinesP_. Zadna z nich nie zwraca przestrzeni Hougha. Wynikiem działania pierwszej jest lista parametrów prostych (krotki zawierające rho, theta). Druga zwraca krotki 4-ro elementowe ze współrzędnymi końców odcinków wykorzystanych do wylicznia parametrów (czyli znalezienia prostej). 
8. Wyznacz linie obecne na obrazie za pomocą funkcji _HoughLines_. Wykryte linie narysuj na obrazie początkowym (UWAGA: wczytanym bez konwersji na graylevel). Do wyświetlania linii wykorzystaj przykładowy kod:

In [ ]:
img = cv2.imread('kwadraty.png')
out = img.copy()

lines = cv2.HoughLines(edges, 1, np.pi / 180, 66)

if lines is not None:
    for line in lines:
        rho, theta_cv = line[0]

        a = np.cos(theta_cv)
        b = np.sin(theta_cv)
        x0 = a * rho
        y0 = b * rho
        x1 = int(x0 + 1000 * (-b))
        y1 = int(y0 + 1000 * (a))
        x2 = int(x0 - 1000 * (-b))
        y2 = int(y0 - 1000 * (a))

        cv2.line(out, (x1, y1), (x2, y2), (0, 0, 255), 2)

plt.figure(figsize=(6, 6))
plt.imshow(cv2.cvtColor(out, cv2.COLOR_BGR2RGB))
plt.axis('off')
plt.show()


9. Wyznacz odcinki obecne na obrazie za pomocą funkcji _HoughLinesP_. Wykryte odcinki narysuj na obrazie początkowym (UWAGA: wczytanym bez konwersji na graylevel). 

In [ ]:
img = cv2.imread('kwadraty.png')
out = img.copy()

linesP = cv2.HoughLinesP(
    edges,
    1,
    np.pi / 180,
    40,
    minLineLength=35,
    maxLineGap=5
)

if linesP is not None:
    for line in linesP:
        x1, y1, x2, y2 = line[0]
        cv2.line(out, (x1, y1), (x2, y2), (0, 255, 0), 2)

plt.figure(figsize=(6, 6))
plt.imshow(cv2.cvtColor(out, cv2.COLOR_BGR2RGB))
plt.axis('off')
plt.show()


### Transformata Hougha dla obrazu rzeczywistego.

Bazując na kodzie stworzonym w punkcie B wyszukamy linie na obrazie rzeczywistym.
   1. Wczytaj obraz "lab112.png". Wyświetl go.
   2. Wykorzystując wszystkie poznane techniki przetwarzania obrazów (filtracja, przekształcenia morfologiczne, binaryzację, detekcję krawędzi) wyodrębnij krawędzie samych kwadratów - tak aby były jak najlepszej jakości (cienkie) - jednocześnie eliminując z obrazu zakłócenia.
   3. Wykorzystaj funkcje *hough_line* i *hough_line_peaks* do detekcji linii na obrazie, a następnie np. wykorzystując kod z punktu 8 poprzedniego opisu narysuj na oryginalnym obrazie znalezione linie.

In [ ]:
img = cv2.imread('lab112.png')
img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)

blur = cv2.GaussianBlur(gray, (5, 5), 0)

_, th = cv2.threshold(blur, 80, 255, cv2.THRESH_BINARY_INV)

num_labels, labels, stats, centroids = cv2.connectedComponentsWithStats(th, connectivity=8)

mask = np.zeros_like(th)
h, w = th.shape

for i in range(1, num_labels):
    x, y, ww, hh, area = stats[i]
    touches_border = (x == 0) or (y == 0) or (x + ww >= w) or (y + hh >= h)

    if area > 5000 and not touches_border:
        mask[labels == i] = 255

edges = cv2.Canny(mask, 50, 150)

H, theta, rho = hough_line(edges)
accums, angles, dists = hough_line_peaks(
    H, theta, rho,
    threshold=0.3 * np.max(H),
    num_peaks=8
)

out = img.copy()

for angle, dist in zip(angles, dists):
    a = np.cos(angle)
    b = np.sin(angle)
    x0 = a * dist
    y0 = b * dist
    x1 = int(x0 + 1000 * (-b))
    y1 = int(y0 + 1000 * (a))
    x2 = int(x0 - 1000 * (-b))
    y2 = int(y0 - 1000 * (a))

    cv2.line(out, (x1, y1), (x2, y2), (0, 0, 255), 2)

fig, ax = plt.subplots(1, 5, figsize=(20, 5))

ax[0].imshow(img_rgb)
ax[0].set_title('Oryginal')
ax[0].axis('off')

ax[1].imshow(th, cmap='gray')
ax[1].set_title('Binaryzacja')
ax[1].axis('off')

ax[2].imshow(mask, cmap='gray')
ax[2].set_title('Maska kwadratow')
ax[2].axis('off')

ax[3].imshow(edges, cmap='gray')
ax[3].set_title('Krawedzie')
ax[3].axis('off')

ax[4].imshow(cv2.cvtColor(out, cv2.COLOR_BGR2RGB))
ax[4].set_title('Wykryte linie')
ax[4].axis('off')

plt.show()


4. Wczytaj obraz "dom.png". Wypróbuj działanie transformacji Hougha na tym obrazie z wykorzystaniem funkcji _cv2.HoughLinesP_  (oczywiście po odpowiednich przekształceniach). Postaraj się tak przygotować obraz z krawędziami i dobrać parametry aby narysować na oryginalnym obrazie odcinki obejmujące zarysy domu. Weź pod uwage dodatkowe parametry funkcji, takie jak:   minLineLength, maxLineGap.

In [ ]:
dom = cv2.imread('dom.png', cv2.IMREAD_GRAYSCALE)
plt.imshow(dom, 'gray'), plt.title('Oryginał'), plt.axis('off')
plt.show()

dombgr = cv2.imread('dom.png')

dom_blur = cv2.GaussianBlur(dom, (5, 5), 0)

dom_edges = cv2.Canny(dom_blur, 90, 180)

plt.imshow(dom_edges, 'gray'), plt.title('Krawędzie domu'), plt.axis('off')
plt.show()

linesP_dom = cv2.HoughLinesP(
            dom_edges,
            1,
            np.pi/180,
            threshold=20,
            minLineLength=40,
            maxLineGap=10
            )

for points in linesP_dom:
    x1,y1,x2,y2=points[0]
    cv2.line(dombgr,(x1,y1),(x2,y2),(255,0,0),2)

plt.imshow(dombgr), plt.axis('off'), plt.title('Linie z transformacji Hougha')
plt.show()
